In [ ]:
import os
from pathlib import Path

ROOT_IMAGES = []
ROOT_IMAGES.append(Path('D:\\Google Photos'))
ROOT_IMAGES.append(Path('C:\\Users\\nshel\\iCloudPhotos\\Photos'))

# Get list of image files in directory recursively
image_extensions = ('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff')
image_files = []

def find_images(path):
    for entry in os.scandir(path):
        if entry.is_file() and entry.name.lower().endswith(image_extensions):
            image_files.append(os.path.join(path, entry.name))
        elif entry.is_dir():
            find_images(entry.path)

for dir in ROOT_IMAGES:
    find_images(dir)

print("Image files found:", len(image_files))


from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS
from random import shuffle

# shuffle(image_files)

image_data = []
i = 0
for img in image_files:
    # Open image and extract EXIF data
    with Image.open(img) as image:
        try:
            exif = image._getexif()
        except AttributeError:
            print("No EXIF data found in image " + img)
            exif = None

        if exif is not None:
            # Create dictionary of tag names and values
            exif_data = {}
            for tag_id in exif:
                tag = TAGS.get(tag_id, tag_id)
                data = exif[tag_id]
                # Special handling for GPS data
                if tag == "GPSInfo":
                    gps_data = {}
                    for gps_tag in data:
                        sub_tag = GPSTAGS.get(gps_tag, gps_tag)
                        gps_data[sub_tag] = data[gps_tag]
                    data = gps_data

                # Convert GPS coordinates to decimal degrees if GPS data is present
                if tag == "GPSInfo":
                    # Check if we have the required GPS coordinates
                    if "GPSLatitude" in data and "GPSLongitude" in data:
                        # Extract latitude
                        lat_deg = data["GPSLatitude"][0]
                        lat_min = data["GPSLatitude"][1]
                        lat_sec = data["GPSLatitude"][2]
                        latitude = lat_deg + (lat_min / 60.0) + (lat_sec / 3600.0)

                        # Apply latitude reference (N/S)
                        if data.get("GPSLatitudeRef", "N") == "S":
                            latitude = -latitude

                        # Extract longitude
                        lon_deg = data["GPSLongitude"][0]
                        lon_min = data["GPSLongitude"][1]
                        lon_sec = data["GPSLongitude"][2]
                        longitude = lon_deg + (lon_min / 60.0) + (lon_sec / 3600.0)

                        # Apply longitude reference (E/W)
                        if data.get("GPSLongitudeRef", "E") == "W":
                            longitude = -longitude

                        # Add converted coordinates to the GPS data
                        exif_data["latitude"] = latitude
                        exif_data["longitude"] = longitude
                else:
                    exif_data[tag] = data

            camera = ""
            if 'Make' in exif_data.keys():
                camera = exif_data['Make']
            if 'Model' in exif_data.keys():
                camera += " " + exif_data['Model']

            if 'latitude' not in exif_data.keys():
                exif_data['latitude'] = 0
            if 'longitude' not in exif_data.keys():
                exif_data['longitude'] = 0
            if 'DateTimeDigitized' not in exif_data.keys():
                if 'DateTimeOriginal' in exif_data.keys():
                    exif_data['DateTimeDigitized'] = exif_data['DateTimeOriginal']
                else:
                    exif_data['DateTimeDigitized'] = "none"
                    print("No DateTime!")

            # print("--------------------------------")
            # print(camera)
            # print('"' + img + '"')
            # print(exif_data['DateTimeDigitized'])
            # print(exif_data['DateTimeOriginal'])
            # print(exif_data['latitude'], exif_data['longitude'])

            image_data.append({
                'image_path': img,
                'camera': camera,
                'latitude': exif_data['latitude'],
                'longitude': exif_data['longitude'],
                'timestamp': exif_data['DateTimeDigitized'],
            })
            print(img)
        else:
            print("\nNo EXIF data found in image")
    i+=1


: 

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime
import numpy as np

# Convert valid timestamps to datetime objects
valid_dates = []
for img in image_data:
    if img['timestamp'] != "none":
        # Parse ISO format timestamp, strip timezone
        dt = datetime.fromisoformat(img['timestamp'].replace(".000-08:00",""))
        valid_dates.append(dt)

# Create histogram
plt.figure(figsize=(12,6))
plt.hist(valid_dates, bins=50, edgecolor='black')
plt.title('Distribution of Photos Over Time')
plt.xlabel('Date')
plt.ylabel('Number of Photos')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
from datetime import datetime
import pytz

# Function to convert EXIF datetime format to ISO format with timezone
def convert_datetime(exif_datetime_str, timezone_str="-08:00"):  # Default to PST
    if exif_datetime_str == "none":
        return "none"
        
    try:
        # Parse EXIF datetime format
        dt = datetime.strptime(exif_datetime_str, "%Y:%m:%d %H:%M:%S")
        
        # Format as ISO8601 with timezone
        formatted = dt.strftime("%Y-%m-%dT%H:%M:%S.000") + timezone_str
        return formatted
    except:
        return "none"

# Convert timestamps in image_data
for img in image_data:
    img['timestamp'] = convert_datetime(img['timestamp'])


In [ ]:
import pandas as pd

# Convert image_data list of dicts to DataFrame
df = pd.DataFrame(image_data)

# Display as formatted table
print("\nImage Data Summary:")
print(df.to_string(index=False))


In [ ]:
# Filter out entries where timestamp is "none"
filtered_data = [img for img in image_data if img['timestamp'] != "none"]

print(f"\nTotal images: {len(image_data)}")
print(f"Images with timestamps: {len(filtered_data)}")

# Update samples for use in next cell
samples = filtered_data


In [ ]:
import sqlite3

db_path = 'photos.db'

# Delete database if it exists
import os
if os.path.exists(db_path):
    os.remove(db_path)

def initialize_database(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS photos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            time TEXT,
            camera TEXT,
            imgpath TEXT,
            lat Decimal(8,6),
            lng Decimal(9,6)
        )
    ''')
    conn.commit()
    return conn

conn = initialize_database(db_path)

cursor = conn.cursor()
for sample in samples:
    cursor.execute('''
      INSERT INTO photos (time, lat, lng, camera, imgpath)
      VALUES (?, ?, ?, ?, ?)
      ''', (sample['timestamp'], sample['latitude'], sample['longitude'], sample['camera'], sample['image_path']))
conn.commit()



In [ ]:
import sqlite3
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT * FROM photos LIMIT 5")
rows = cursor.fetchall()
for row in rows:
    print(row)
